# 🛡️ Surface Code — Kuantum Hata Düzeltme Demo

Bu not defteri, Surface Code'un temel kavramlarını Python ile gösterir.

**İçindekiler:**
1. Surface Code nedir?
2. Izgara yapısı ve qubit türleri
3. Hata enjeksiyonu ve sendrom ölçümü
4. Görselleştirme
5. Basit hata düzeltme simülasyonu
6. Qiskit ile tekrar (repetition) kodu

---

In [ ]:
!pip install qiskit qiskit-aer pylatexenc -q
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
print('Kurulum tamam ✓')

## 1. Surface Code Nedir?

Surface Code bir **emülatör veya yazılım değildir**. Fiziksel qubit'lerin 2D ızgara üzerinde nasıl düzenleneceğini belirleyen bir **hata düzeltme kodudur**.

İki tür qubit vardır:
- **Veri qubitleri**: Hesaplama bilgisini taşır
- **Ölçüm (ancilla) qubitleri**: Komşularını kontrol eder

Anahtar fikir: Ölçüm qubitleri **asıl bilgiyi ölçmez**, sadece komşular arasında tutarsızlık olup olmadığını kontrol eder. Bu sayede süperpozisyon korunur.

## 2. Izgara Yapısını Görselleştirme

In [ ]:
def draw_surface_code(grid_size=5, errors=None, syndromes=None, title='Surface Code Izgarası'):
    """Surface Code ızgarasını çiz.
    errors: hatalı veri qubit koordinatları [(r,c), ...]
    syndromes: alarm veren ölçüm qubit koordinatları [(r,c), ...]
    """
    if errors is None: errors = []
    if syndromes is None: syndromes = []

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-0.5, grid_size - 0.5)
    ax.set_ylim(-0.5, grid_size - 0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.invert_yaxis()

    # Bağlantı çizgileri
    for r in range(grid_size):
        for c in range(grid_size):
            is_data = (r + c) % 2 == 0
            if not is_data:
                # Ölçüm qubiti -> komşu veri qubitlerine çizgi
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < grid_size and 0 <= nc < grid_size:
                        is_error_line = (nr,nc) in errors and (r,c) in syndromes
                        ax.plot([c, nc], [r, nr],
                               color='#F59E0B' if is_error_line else '#CBD5E1',
                               lw=3 if is_error_line else 1,
                               alpha=0.8 if is_error_line else 0.3,
                               zorder=0)

    # Qubit'leri çiz
    for r in range(grid_size):
        for c in range(grid_size):
            is_data = (r + c) % 2 == 0
            if is_data:
                has_error = (r, c) in errors
                color = '#EF4444' if has_error else '#0891B2'
                marker = 's'
                size = 600
                label = f'HATA' if has_error else 'V'
            else:
                has_syndrome = (r, c) in syndromes
                color = '#F59E0B' if has_syndrome else '#E2E8F0'
                marker = 'o'
                size = 400
                label = '⚠' if has_syndrome else 'Ö'

            ax.scatter(c, r, c=color, s=size, marker=marker, zorder=2,
                      edgecolors='#1E293B', linewidths=1.5)
            ax.text(c, r, label, ha='center', va='center', fontsize=8,
                   fontweight='bold', color='white' if is_data else '#1E293B', zorder=3)

    # Lejant
    legend_items = [
        mpatches.Patch(color='#0891B2', label='Veri Qubit (hesap yapar)'),
        mpatches.Patch(color='#E2E8F0', label='Ölçüm Qubit (hata arar)'),
        mpatches.Patch(color='#EF4444', label='Hatalı Qubit'),
        mpatches.Patch(color='#F59E0B', label='Alarm (sendrom)'),
    ]
    ax.legend(handles=legend_items, loc='upper right', fontsize=9)
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.grid(True, alpha=0.1)
    plt.tight_layout()
    plt.show()

# Temiz ızgara
draw_surface_code(5, title='Surface Code Izgarası (5×5) — Hatasız')
print('V = Veri qubit (kare), Ö = Ölçüm qubit (daire)')
print(f'Toplam: 13 veri + 12 ölçüm = 25 fiziksel qubit → ~1 mantıksal qubit')

## 3. Hata Enjeksiyonu ve Sendrom Ölçümü

Bir veri qubit'inde hata oluştuğunda, komşu ölçüm qubitleri bunu tespit eder.

In [ ]:
def compute_syndromes(grid_size, errors):
    """Hatalı veri qubitlerinden sendrom (alarm) hesapla."""
    syndromes = []
    for r in range(grid_size):
        for c in range(grid_size):
            is_data = (r + c) % 2 == 0
            if not is_data:
                # Komşu veri qubitlerini kontrol et
                error_count = 0
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < grid_size and 0 <= nc < grid_size:
                        if (nr, nc) in errors:
                            error_count += 1
                # Tek sayıda hatalı komşu = alarm!
                if error_count % 2 == 1:
                    syndromes.append((r, c))
    return syndromes

# Tek bir hata
errors = [(2, 2)]  # Ortadaki veri qubit'inde hata
syndromes = compute_syndromes(5, errors)
print(f'Hata konumu: {errors}')
print(f'Alarm veren ölçüm qubitleri: {syndromes}')
print(f'Alarm sayısı: {len(syndromes)}')
draw_surface_code(5, errors, syndromes, 'Tek Hata → Sendrom Tespiti')

In [ ]:
# İki hata (komşu)
errors_2 = [(2, 2), (2, 4)]
syndromes_2 = compute_syndromes(5, errors_2)
print(f'Hatalar: {errors_2}')
print(f'Alarm veren ölçüm qubitleri: {syndromes_2}')
draw_surface_code(5, errors_2, syndromes_2, 'İki Hata → Sendrom Deseni')
print('Dikkat: İki hatanın arasındaki ölçüm qubit ALARM VERMEYEBİLİR')
print('(çift sayıda hatalı komşu = alarm yok → hatalar birbirini maskeler)')

## 4. Adım Adım Hata Düzeltme Süreci

In [ ]:
# 4 adımlı süreç görselleştirmesi
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

titles = ['Adım 1: Sağlıklı Sistem', 'Adım 2: Hata Oluştu!',
          'Adım 3: Sendrom Tespiti', 'Adım 4: Düzeltildi!']
error_sets = [[], [(2,2)], [(2,2)], []]
syndrome_sets = [[], [], [(1,2),(2,1),(2,3),(3,2)], []]

grid_size = 5
for idx, (ax, title, errs, syns) in enumerate(zip(axes, titles, error_sets, syndrome_sets)):
    ax.set_xlim(-0.5, grid_size-0.5)
    ax.set_ylim(-0.5, grid_size-0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12, fontweight='bold',
                color=['#0891B2','#EF4444','#F59E0B','#10B981'][idx])
    ax.invert_yaxis()

    for r in range(grid_size):
        for c in range(grid_size):
            is_data = (r+c)%2==0
            if is_data:
                has_err = (r,c) in errs
                color = '#EF4444' if has_err else '#0891B2'
                ax.scatter(c, r, c=color, s=250, marker='s', edgecolors='#1E293B', lw=1)
            else:
                has_syn = (r,c) in syns
                color = '#F59E0B' if has_syn else '#E2E8F0'
                ax.scatter(c, r, c=color, s=180, marker='o', edgecolors='#94A3B8', lw=1)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Surface Code Hata Düzeltme Süreci', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Hata Düzeltme Simülasyonu

Rastgele hatalar oluşturup düzeltme başarı oranını ölçelim.

In [ ]:
def simple_correction(grid_size, errors):
    """Basit minimum weight matching benzeri düzeltme."""
    syndromes = compute_syndromes(grid_size, errors)
    if not syndromes:
        return True  # Hata yok veya tespit edilemedi

    # Her sendrom çifti arasındaki en olası hata konumunu bul
    corrected_errors = set()
    for sr, sc in syndromes:
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = sr+dr, sc+dc
            if 0 <= nr < grid_size and 0 <= nc < grid_size:
                if (nr+nc)%2 == 0:  # Veri qubit
                    corrected_errors.add((nr, nc))

    # Düzeltme başarılı mı?
    remaining = set(errors).symmetric_difference(corrected_errors)
    # Kalan hata çift sayıysa mantıksal hata yok
    return len(remaining) % 2 == 0

# Simülasyon
grid_size = 5
n_trials = 1000
error_rates = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
success_rates = []

# Veri qubit konumları
data_qubits = [(r,c) for r in range(grid_size) for c in range(grid_size) if (r+c)%2==0]

for p in error_rates:
    successes = 0
    for _ in range(n_trials):
        # Rastgele hatalar oluştur
        errors = [q for q in data_qubits if np.random.random() < p]
        if simple_correction(grid_size, errors):
            successes += 1
    success_rates.append(successes / n_trials)

# Grafik
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(error_rates, success_rates, 'o-', color='#0891B2', lw=2, markersize=8)
ax.axhline(y=0.99, color='#10B981', ls='--', lw=1, alpha=0.7, label='%99 hedef')
ax.axvline(x=0.01, color='#EF4444', ls='--', lw=1, alpha=0.7, label='~%1 hata esigi')
ax.set_xlabel('Fiziksel Hata Oranı (p)', fontsize=12)
ax.set_ylabel('Düzeltme Başarı Oranı', fontsize=12)
ax.set_title('Surface Code (5×5) Hata Düzeltme Performansı', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.02)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Fiziksel hata oranı < %1 olduğunda Surface Code çok etkili!')
print('Hata oranı arttıkça düzeltme gücü hızla düşer.')

## 6. Qiskit ile Tekrar (Repetition) Kodu

Surface Code'un 1 boyutlu basit versiyonu: **Repetition Code**.
3 fiziksel qubit → 1 mantıksal qubit (oylama sistemi).

In [ ]:
# 3 qubitli repetition code
# 1 mantıksal qubit = 3 fiziksel qubit + 2 ölçüm qubit

def repetition_code_circuit(initial_state='0', inject_error=None):
    """
    3-qubit repetition code.
    q0, q1, q2: veri qubitleri
    q3, q4: ölçüm (ancilla) qubitleri
    """
    qc = QuantumCircuit(5, 4)  # 5 qubit, 4 klasik bit

    # Mantıksal |1⟩ hazırla (istenmişse)
    if initial_state == '1':
        qc.x([0, 1, 2])  # Üç kopyaya da X uygula

    qc.barrier(label='Hazırlık')

    # Hata enjeksiyonu
    if inject_error is not None:
        qc.x(inject_error)  # Bit-flip hatası
        qc.barrier(label=f'Hata q{inject_error}')

    # Sendrom ölçümü
    # Ancilla 3: q0 XOR q1
    qc.cx(0, 3)
    qc.cx(1, 3)
    # Ancilla 4: q1 XOR q2
    qc.cx(1, 4)
    qc.cx(2, 4)

    qc.barrier(label='Sendrom')

    # Ölçüm
    qc.measure(3, 0)  # sendrom 1
    qc.measure(4, 1)  # sendrom 2

    qc.barrier(label='Düzeltme')

    # Veri qubitlerini ölç
    qc.measure(0, 2)
    qc.measure(1, 3)

    return qc

# Hatasız durum
qc_clean = repetition_code_circuit('0')
print('=== Hatasız Devre ===')
display(qc_clean.draw('mpl', style='iqp'))
sv = Statevector.from_label('00000').evolve(QuantumCircuit(5))  # Başlangıç
print('Beklenen: Sendrom = 00 (hata yok)')

In [ ]:
# Hatalı durum (q1'de bit-flip)
qc_error = repetition_code_circuit('0', inject_error=1)
print('=== q1 de Hata Olan Devre ===')
display(qc_error.draw('mpl', style='iqp'))
print('Beklenen: Sendrom = 11 → q1 de hata tespit edildi!')
print('\nSendrom tablosu:')
print('  00 → Hata yok')
print('  10 → q0 da hata')
print('  11 → q1 de hata')
print('  01 → q2 de hata')

In [ ]:
# Sendrom tablosunu görselleştir
fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

table_data = [
    ['Sendrom (s1,s2)', 'Anlam', 'Düzeltme'],
    ['00', 'Hata yok', 'Bir sey yapma'],
    ['10', 'q0 ≠ q1 ve q1 = q2', 'q0 ya X uygula'],
    ['11', 'q0 ≠ q1 ve q1 ≠ q2', 'q1 e X uygula'],
    ['01', 'q0 = q1 ve q1 ≠ q2', 'q2 ye X uygula'],
]

colors = [['#0F172A']*3] + [['#ECFDF5' if r[1]=='Hata yok' else '#FEE2E2']*3 for r in table_data[1:]]
text_colors = [['white']*3] + [['#1E293B']*3]*4

table = ax.table(cellText=table_data, cellColours=colors, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for (row, col), cell in table.get_celld().items():
    cell.set_text_props(color=text_colors[row][col], fontweight='bold' if row==0 else 'normal')
    cell.set_edgecolor('#CBD5E1')

ax.set_title('3-Qubit Repetition Code Sendrom Tablosu', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()
print('Bu, Surface Code un 1 boyutlu hali. Gerçek Surface Code 2D ızgara kullanır.')

## 7. Ölçek Karşılaştırması

In [ ]:
# Surface Code ölçekleme
distances = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
data_per_d = [d**2 + (d-1)**2 for d in distances]  # Yaklaşık
total_per_d = [(2*d-1)**2 for d in distances]  # d×d veri + ölçüm
logical_qubits = [1]*len(distances)  # Her biri 1 mantıksal qubit

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: fiziksel qubit sayısı vs code distance
ax = axes[0]
ax.bar(range(len(distances)), total_per_d, color='#0891B2', alpha=0.7, edgecolor='#0E7490')
ax.set_xticks(range(len(distances)))
ax.set_xticklabels([f'd={d}' for d in distances], rotation=45)
ax.set_ylabel('Toplam Fiziksel Qubit', fontsize=12)
ax.set_title('1 Mantıksal Qubit İçin\nGereken Fiziksel Qubit Sayısı', fontsize=13, fontweight='bold')
for i, v in enumerate(total_per_d):
    ax.text(i, v + 10, str(v), ha='center', fontsize=8, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Sağ: hata bastırma
ax = axes[1]
p_phys = 0.001  # %0.1 fiziksel hata
p_threshold = 0.01
logical_error_rates = [(p_phys/p_threshold)**(d//2 + 1) for d in distances]
ax.semilogy(distances, logical_error_rates, 'o-', color='#10B981', lw=2, markersize=8)
ax.axhline(y=1e-15, color='#EF4444', ls='--', alpha=0.5, label='RSA kırma hedefi')
ax.set_xlabel('Code Distance (d)', fontsize=12)
ax.set_ylabel('Mantıksal Hata Oranı', fontsize=12)
ax.set_title(f'Hata Bastırma (p_fiziksel = {p_phys})', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'\nd=3 → {(2*3-1)**2} fiziksel qubit → 1 mantıksal qubit')
print(f'd=21 → {(2*21-1)**2} fiziksel qubit → 1 mantıksal qubit')
print(f'\nDistance arttıkça mantıksal hata ÜSTEL olarak düşer!')

---
## 📝 Özet

1. **Surface Code** = 2D ızgara üzerinde qubit düzenleme planı (emülatör DEĞİL)
2. **Veri qubitleri** hesap yapar, **ölçüm qubitleri** hata arar
3. Ölçüm qubitleri **asıl bilgiyi bozmadan** komşulardaki tutarsızlığı tespit eder
4. **Sendrom** = hata haritası → klasik bilgisayar analiz eder → düzeltme komutu
5. Fiziksel hata oranı < %1 olmalı (eşik değer)
6. Distance (d) arttıkça hata koruması üstel olarak güçlenir
7. Bedeli: ~1000 fiziksel qubit = 1 mantıksal qubit

🔗 **İleri Okuma:**
- [Fowler et al., Surface codes: Towards practical large-scale quantum computation](https://arxiv.org/abs/1208.0928)
- [Google Quantum AI - Error Correction](https://quantumai.google/research/error-correction)